# Structured and staged Hessian benchmark

This Colab notebook evaluates legacy two-solve staging and the new single-solve structured-GN → exact-Hessian hybrid independently of the matrix-exponential ablation.

It compares six strategies with `TWIN4BUILD_MATRIX_EXP=ss`:

1. always legacy Gauss–Newton (objective curvature only);
2. always structured Gauss–Newton (exact constraint curvature);
3. always exact Hessian;
4. legacy staged feasible/stall switching;
5. legacy staged cost-aware switching;
6. guarded in-place structured-GN → exact switching.

The misleading cross-run gross callback `rho` calibration is disabled. The notebook reports objective/constraint/Jacobian evaluation counts, callback wall time, switch iteration, and Hessian-specific timing instead.

All raw results remain visible—including a fast staged arm with a lower objective—but objective and RMSE comparisons are valid only after strict primal, dual, and collocation-defect qualification.

Use **Runtime → Change runtime type → GPU**, then **Run all**. A fresh runtime and an A100-class GPU are recommended for FP64.

In [ ]:
import warnings

warnings.filterwarnings("ignore", message="Failed to parse namespace")
warnings.filterwarnings("ignore", message="Failed to parse ontology namespace")
warnings.filterwarnings("ignore", message='Neither "df", "filename", nor "uuid"')

TWIN4BUILD_REF = "feature/issue-122/cuda-graph-hessian"

try:
    import twin4build as tb
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}",
        ],
        check=True,
    )
    import twin4build as tb

import inspect
import os
import platform
import time

import numpy as np
import pandas as pd
import torch
import twin4build.estimator._transcription as _tr

_tr_src = inspect.getsource(_tr)
_missing = [
    name
    for name in (
        "hessian_stages",
        "hessian_hybrid",
        "structured_gauss_newton",
        "ipopt_diagnostics",
    )
    if name not in _tr_src
]
if _missing:
    raise RuntimeError(
        "The installed Twin4Build is stale and missing: " + ", ".join(_missing)
        + f"""
Installed at: {tb.__file__}
Install the staged branch, then disconnect/delete and recreate the Colab runtime:
    pip install -q --force-reinstall --no-deps git+https://github.com/JBjoernskov/Twin4Build.git@{TWIN4BUILD_REF}"""
    )

if not torch.cuda.is_available():
    raise RuntimeError("GPU required: Runtime > Change runtime type > GPU")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
print("CPU:", platform.processor() or platform.machine(), "cores:", os.cpu_count())
print("Torch:", torch.__version__)
print("Twin4Build:", tb.__file__)
print("Ref:", TWIN4BUILD_REF)

os.environ["TWIN4BUILD_CUDA_GRAPH"] = "0"
os.environ["TWIN4BUILD_HESS_CHUNK_DIV"] = "1"
os.environ["TWIN4BUILD_COMBINED_HESSIAN"] = "1"
os.environ["TWIN4BUILD_MATRIX_EXP"] = "ss"

In [ ]:
import datetime
import importlib.util as _ilu
import pathlib

from dateutil import tz

import twin4build.examples as _ex_pkg
import twin4build.examples.utils as utils

_path = pathlib.Path(_ex_pkg.__file__).parent / "full_workflow_example.py"
_spec = _ilu.spec_from_file_location("_fwe_staged_hessian", _path)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
fcn = _mod.fcn

STEP = 1200
N_WARMUP = 20
START = [datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))]
FULL_END = [datetime.datetime(2023, 12, 7, tzinfo=tz.gettz("Europe/Copenhagen"))]


def build_model(tag):
    model = tb.Model(id=tag)
    model.load(
        semantic_model_filename=utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc, cc = c["office_temperature_heating_controller"], c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


print("Model builders ready")

In [ ]:
STAGE2_MAXITER = 300

ARMS = [
    ("always_gn", {
        "gauss_newton": True,
        "exact_hessian": False,
        "early_stopping": True,
    }),
    ("always_structured_gn", {
        "structured_gauss_newton": True,
        "early_stopping": True,
    }),
    ("always_exact", {
        "gauss_newton": True,
        "exact_hessian": True,
        "early_stopping": False,
    }),
    # Legacy two-solve arms remain explicit comparators: their raw objective can
    # be useful, but is only solution-quality evidence when KKT/defect-qualified.
    ("staged_stall", {
        "hessian_stages": {
            "switch_rule": "feasible_stall",
            "stage1_maxiter": 40,
            "min_iterations": 10,
        },
        "early_stopping": False,
    }),
    ("staged_cost", {
        "hessian_stages": {
            "switch_rule": "cost_aware",
            "stage1_maxiter": 40,
            "min_iterations": 10,
            "probe_interval": 5,
            "cost_ratio": 3.0,
            "exact_phase_iterations": 6,
        },
        "early_stopping": False,
    }),
    ("hybrid_guarded", {
        "hessian_hybrid": {
            "min_iterations": 5,
            "hard_max_iterations": 12,
            "feas_tol": 1e-3,
            "contraction_window": 4,
            "contraction_threshold": 0.8,
        },
        "early_stopping": False,
    }),
]

rows = []
rmse_rows = []
for label, arm_options in ARMS:
    print(f"\nRunning {label} ...", flush=True)
    effective_options = dict(arm_options)
    if "hessian_stages" in effective_options:
        effective_options["hessian_stages"] = dict(
            effective_options["hessian_stages"]
        )

    # Do not calibrate the legacy cost rule from gross Hessian callback time.
    # That ratio mixes different iterates and double-counts cached Jacobian work;
    # callback and full-solver counters are reported below instead.
    if label == "staged_cost":
        print("  legacy cost_ratio=3.0 (gross callback-rho calibration disabled)")

    model = estimator = None
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    try:
        model = build_model(f"staged_{label}")
        estimator = tb.Estimator(tb.Simulator(model))
        result = estimator.estimate(
            START,
            FULL_END,
            STEP,
            build_parameters(model),
            build_measurements(model),
            n_warmup=N_WARMUP,
            method=("casadi", "ipopt", "ad", "collocation"),
            options=dict(
                maxiter=STAGE2_MAXITER,
                boundary_state_init="auto",
                **effective_options,
            ),
        )
        elapsed = time.perf_counter() - started
        audit = result.get("transcription_audit", {})
        diag = audit.get("ipopt_diagnostics", {})
        stages = audit.get("hessian_stages", {})
        hybrid = audit.get("hessian_hybrid", {})
        callbacks = hybrid.get(
            "hessian_callbacks",
            stages.get("hessian_callbacks", audit.get("hessian_callbacks", {})),
        )
        solver_counters = diag.get("solver_counters", {})
        callback_stats = diag.get("callback_stats", {})
        inf_pr, inf_du = diag.get("inf_pr"), diag.get("inf_du")
        feasibility_qualified = (
            inf_pr is not None
            and inf_pr <= 1e-6
            and float(audit.get("max_defect", np.inf)) <= 1e-6
        )
        strict_converged = (
            diag.get("status") == "Solve_Succeeded"
            and feasibility_qualified
            and inf_du is not None
            and inf_du <= 1e-6
        )
        rows.append({
            "arm": label,
            "seconds": elapsed,
            "strict_converged": strict_converged,
            "feasibility_qualified": feasibility_qualified,
            "return_status": diag.get("status", audit.get("return_status")),
            "iterations_total": result.get("iterations"),
            "objective": result.get("final_objective"),
            "max_defect": audit.get("max_defect", np.nan),
            "inf_pr": inf_pr,
            "inf_du": inf_du,
            "mu": diag.get("mu"),
            "regularization": diag.get("regularization"),
            "stage1_iterations": stages.get("stage1", {}).get("iterations"),
            "stage2_iterations": stages.get("stage2", {}).get("iterations"),
            "stage1_seconds": stages.get("stage1", {}).get("seconds"),
            "stage2_seconds": stages.get("stage2", {}).get("seconds"),
            "switch_iteration": hybrid.get("switch_iteration"),
            "switch_reason": hybrid.get("switch_reason", stages.get("switch_reason")),
            "dual_warm_start": stages.get("stage2", {}).get("warm_start_duals"),
            "gn_hessian_calls": callbacks.get("gauss_newton", {}).get("calls"),
            "structured_gn_calls": callbacks.get("structured_gauss_newton", {}).get("calls"),
            "structured_gn_seconds": callbacks.get("structured_gauss_newton", {}).get("seconds"),
            "exact_hessian_calls": callbacks.get("exact", {}).get("calls"),
            "exact_hessian_seconds": callbacks.get("exact", {}).get("seconds"),
            "objective_calls": solver_counters.get("n_call_nlp_f"),
            "constraint_calls": solver_counters.get("n_call_nlp_g"),
            "jacobian_calls": solver_counters.get("n_call_nlp_jac_g"),
            "callback_wall_seconds": sum(
                item.get("seconds", 0.0) for item in callback_stats.values()
            ),
            "peak_cuda_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
            "error": None,
        })
        for sensor, values in audit.get("per_sensor", {}).items():
            rmse_rows.append({
                "arm": label,
                "strict_converged": strict_converged,
                "sensor": sensor,
                "nlp_rmse": float(values["nlp_rmse"]),
                "rollout_rmse": float(values["rollout_rmse"]),
                "do_step_rmse": float(values["do_step_rmse"]),
            })
    except Exception as exc:
        rows.append({
            "arm": label,
            "seconds": time.perf_counter() - started,
            "strict_converged": False,
            "return_status": "FAILED",
            "error": f"{type(exc).__name__}: {str(exc).splitlines()[0]}",
        })
        print("  FAILED:", rows[-1]["error"])
    finally:
        del estimator, model
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
rmse_results = pd.DataFrame(rmse_rows)
with pd.option_context("display.max_columns", None, "display.width", 300):
    print("All raw runs (objective/time are diagnostic when convergence is false)")
    display(results)
    print("Strictly converged runs — valid speed/objective comparison")
    display(results[results["strict_converged"] == True])
    print("Per-sensor RMSE (filter strict_converged=True for final comparison)")
    display(rmse_results)